# Import des bibliothèques

In [ ]:
import pandas as pd
import pathlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import gensim
from collections import defaultdict
from gensim import corpora
import re
from gensim.models import LdaModel
import spacy
from spacy.tokenizer import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation

In [28]:

text = open("Corpus_textes/1879_ Goncourt_Les-Freres-Zemganno_N.txt").read()
nlp = spacy.load("fr_core_news_sm")
    

## Tokenisation et nettoyage des données

```python

In [30]:
doc= nlp(text)
tokens = [
    token.text for token in doc if not token.is_stop 
    and not token.is_punct 
    and not token.like_num 
    and not token.is_space]

print(tokens[:20])

['Edmond', 'Goncourt', 'Frères', 'Zemganno', 'MADAME', 'ALPHONSE', 'DAUDET', 'pleine', 'campagne', 'pied', 'poteau', 'octroi', 'dressé', 'carrefour', 'croisaient', 'routes', 'passait', 'château', 'Louis', 'XIII']


## Lementisation

In [31]:
tokens_propres= [
    token.lemma_ for token in doc if not token.is_stop
    and not token.is_punct
    and not token.is_space]

In [32]:
print(tokens_propres[:20])

['Edmond', 'Goncourt', 'frère', 'Zemganno', 'monsieur', 'ALPHONSE', 'DAUDET', 'pleine', 'campagn', 'pied', 'poteau', 'octroi', 'dresser', 'carrefour', 'croiser', 'route', 'passer', 'château', 'Louis', 'xiii']


# Avec Gensim Recherche champs lexicaux sur un seul texte (Gouncourt : Freres Zemganno)

## Sans stop words et avec ponctuation, chiffres et espaces

In [38]:
taille_bloc = 200
documents = [tokens_propres[i:i + taille_bloc] for i in range(0, len(tokens_propres), taille_bloc)]

print(f"texte découpé en {len(documents)} mini-documents.")

id2word = corpora.Dictionary(documents)
corpus = [id2word.doc2bow(doc) for doc in documents]
lda_model = LdaModel(corpus=corpus,    
                     id2word=id2word,  # Dictionnaire pour mapper les IDs aux mots
                     num_topics=3,  # Nombre de thèmes à extraire
                     random_state=42, # Pour la reproductibilité
                     passes=50, # Nombre de passes sur le corpus pour l'entraînement
                     iterations=100) # Nombre d'itérations pour la convergence

print("-Champs Lexicaux trouvés-")
for idx, topic in lda_model.print_topics(-1):
    print(f"Thème {idx}: {topic}")

texte découpé en 111 mini-documents.
-Champs Lexicaux trouvés-
Thème 0: 0.005*"faire" + 0.005*"petit" + 0.005*"frère" + 0.005*"grand" + 0.005*"Gianni" + 0.004*"jour" + 0.004*"Nello" + 0.003*"bien" + 0.003*"être" + 0.003*"chose"
Thème 1: 0.009*"Gianni" + 0.008*"petit" + 0.008*"Nello" + 0.007*"frère" + 0.007*"faire" + 0.004*"jour" + 0.004*"grand" + 0.004*"donner" + 0.004*"homme" + 0.004*"être"
Thème 2: 0.009*"Gianni" + 0.008*"frère" + 0.008*"petit" + 0.007*"Nello" + 0.007*"faire" + 0.006*"jour" + 0.005*"pied" + 0.005*"chose" + 0.004*"dire" + 0.004*"saut"


## Avec stop words,  et sans ponctuation, chiffres et espaces

In [37]:

stopwords = {"nello", "gianni", "faire", "dire", "aller", "bien", "grand", "petit", "choir", "voir"}
tokens_super_propres = [
    token.lemma_ for token in doc 
    if not token.is_stop 
    and token.is_alpha 
    and token.pos_ in ["NOUN", "ADJ"] # garder que les noms et adjectifs
    and token.lemma_ not in stopwords]

taille_bloc = 200
documents = [tokens_super_propres[i:i + taille_bloc] for i in range(0, len(tokens_super_propres), taille_bloc)]

print(f" texte découpé en {len(documents)} mini-documents.")

id2word = corpora.Dictionary(documents)
corpus = [id2word.doc2bow(doc) for doc in documents]
lda_model = LdaModel(corpus=corpus, 
                     id2word=id2word, # mapping entre les mots et leurs ids
                     num_topics=3, # nombre de thèmes à trouver
                     random_state=42, # pour la reproductibilité
                     passes=50,  # nombre de passes sur le corpus
                     iterations=100) # nombre d'itérations pour la convergence

print("-Champs Lexicaux trouvés-")
for idx, topic in lda_model.print_topics(-1):
    print(f"Thème {idx}: {topic}")

 texte découpé en 71 mini-documents.
-Champs Lexicaux trouvés-
Thème 0: 0.013*"frère" + 0.008*"jour" + 0.008*"corps" + 0.008*"main" + 0.007*"chose" + 0.006*"pied" + 0.006*"jeune" + 0.006*"homme" + 0.005*"tête" + 0.005*"jambe"
Thème 1: 0.005*"femme" + 0.005*"homme" + 0.004*"chose" + 0.004*"jour" + 0.004*"frère" + 0.003*"moment" + 0.003*"corps" + 0.003*"voiture" + 0.003*"cheval" + 0.003*"temps"
Thème 2: 0.013*"frère" + 0.009*"jour" + 0.007*"pied" + 0.005*"temps" + 0.005*"chose" + 0.005*"heure" + 0.005*"saut" + 0.005*"fois" + 0.005*"exercice" + 0.005*"moment"


# Utilisation de TF-IDF pour identifier les mots les plus importants

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# On crée un faux corpus avec juste ton texte propre
corpus_test = [" ".join(tokens_super_propres)]

vectorizer = TfidfVectorizer(max_features=20)
tfidf_matrix = vectorizer.fit_transform(corpus_test)

# Récupérer les mots et leurs scores
feature_names = vectorizer.get_feature_names_out()
scores = tfidf_matrix.toarray().flatten()

# Afficher les 20 mots qui définissent vraiment le texte
mots_cles = pd.DataFrame({'mot': feature_names, 'importance': scores}).sort_values(by='importance', ascending=False)
print(mots_cles)

         mot  importance
5      frère    0.502165
9       jour    0.353282
0      chose    0.252344
14      pied    0.249821
10      main    0.237204
1      corps    0.222063
6      homme    0.217016
8      jeune    0.189258
12    moment    0.189258
3      femme    0.184211
16     temps    0.176641
2   exercice    0.171594
18      tour    0.166547
13      oeil    0.156453
19      tête    0.156453
11    milieu    0.148883
4       fois    0.143836
7      jambe    0.141313
15      saut    0.138789
17     terre    0.138789


# Utilsiation de Scikit-learn pour l'analyse de texte

In [39]:
vectorizer = TfidfVectorizer(max_features=25, min_df=1) # On garde les 25 mots les plus importants
tfidf_matrix = vectorizer.fit_transform(tokens_propres)

In [40]:
kmeans = KMeans(n_clusters=3, random_state=42)
cluster = kmeans.fit_predict(tfidf_matrix)

In [41]:
print(cluster)

[0 0 1 ... 0 0 0]


In [42]:
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(tfidf_matrix)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [ ]:
def afficher_topics(model, vectorizer, n_top_words):
    mots = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        message = f"Thème #{topic_idx} : "
        message += " | ".join([mots[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

# On affiche les 10 mots principaux des themes 
afficher_topics(lda, vectorizer, 10)

Thème #0 : nello | faire | grand | chose | bien | homme | donner | coup | venir | temps
Thème #1 : gianni | frère | pied | main | femme | moment | mettre | venir | temps | jeune
Thème #2 : petit | jour | être | cirque | corps | dire | voir | jeune | temps | venir
